# rl_nba — run the cross-sell simulation

This notebook runs the **full reinforcement-learning cross-sell simulation** end to end.
Just run the cells **top to bottom** — it finds the project and config on its own, installs
what it needs, and prints the results plus the learning-curve chart. Nothing to configure.

**Prerequisite:** Jupyter itself (you already have it if you're reading this). Everything
else is handled by the cells below.

## 1 · Locate the project
Finds the code and config no matter which folder you launched Jupyter from, and puts the package on the import path — so this works *without* pip-installing anything.

In [ ]:
import sys
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError(
        f"This project needs Python 3.11+, but this Jupyter kernel is Python "
        f"{sys.version_info.major}.{sys.version_info.minor}. "
        "Switch it via the menu: Kernel > Change Kernel > a 3.11+ environment."
    )


def _find_project_root(start: Path) -> Path:
    """Walk up from `start` looking for the rl_nba package (src/rl_nba)."""
    for base in [start, *start.parents]:
        if (base / "src" / "rl_nba").is_dir():
            return base
        nested = base / "rl_nba"          # in case we're at the monorepo root
        if (nested / "src" / "rl_nba").is_dir():
            return nested
    raise RuntimeError(
        f"Could not find the rl_nba project (looked for src/rl_nba) starting from {start}. "
        "Open this notebook from inside the repository."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))          # makes `import rl_nba` work without installing

# The config lives in the monorepo-root config/ folder (a sibling of the project).
CONFIG_PATH = next(
    (c for c in [PROJECT_ROOT.parent / "config" / "rl_nba_config.yml",
                 PROJECT_ROOT / "config" / "rl_nba_config.yml"] if c.is_file()),
    None,
)
if CONFIG_PATH is None:
    raise FileNotFoundError(f"Could not find config/rl_nba_config.yml near {PROJECT_ROOT}")
REQUIREMENTS = PROJECT_ROOT / "requirements.txt"

print("project root :", PROJECT_ROOT)
print("source path  :", SRC, "(added to sys.path)")
print("config file  :", CONFIG_PATH)

## 2 · Install dependencies (run once)
Installs numpy / pandas / matplotlib / pyyaml / pyarrow into this kernel. Safe to re-run — it's a no-op if they're already present. *If an import below ever fails, restart the kernel and run the cells again.*

In [ ]:
import subprocess
import sys

print("Installing runtime dependencies (skipped if already present)…")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQUIREMENTS)],
    check=True,
)
print("Done.")

## 3 · Load the configuration
One YAML file drives everything: the products the agent can offer, what counts as reward, the customer schema, and which models to compare.

In [ ]:
from rl_nba.config import load_config

config = load_config(CONFIG_PATH)
print("Products (the actions):", ", ".join(config.products.catalog))
print("Reward signal         :", config.reward.type)
print("Rounds per agent      :", f"{config.experiment.n_rounds:,}")
print("Models compared       :", ", ".join(config.experiment.agents))

## 4 · Build the world and peek at the data
`prepare_experiment` generates the (synthetic) customers and builds the simulated environment the agents will learn about. Below that is a sample of the customer table — swap the config's data source to a real export and this is the shape it must match.

In [ ]:
from rl_nba.run import prepare_experiment, describe

prepared = prepare_experiment(config)   # customers + simulated environment
describe(prepared)
prepared.customers.head()

## 5 · Run the full simulation
Every model faces the **same** customers in the same order, so the comparison is fair. `random` is the floor, `baseline` sees only profile + holdings, `enhanced` sees the full customer view. This takes a few seconds.

In [ ]:
from rl_nba.run import run_agents


def show(label, algorithm, n_features, result, seconds):
    print(
        f"  {label:<10} ({algorithm}, {n_features} features)  "
        f"mean reward {result.mean_reward:>10,.2f}   "
        f"cumulative regret {result.cumulative_regret:>12,.0f}   ({seconds:.1f}s)"
    )


print(f"Running {len(config.experiment.agents)} models on the same "
      f"{config.experiment.n_rounds:,} customers…\n")
results = run_agents(prepared, on_agent=show)

## 6 · Results table
Higher mean reward and lower regret are better. `uplift_vs_random_pct` is how much more value each model earns than offering at random.

In [ ]:
from rl_nba.evaluation import summarize_results

summary = summarize_results(results)
summary

## 7 · Learning curves
Left: average reward climbing toward the dashed *oracle* (the best achievable). Right: cumulative regret — flatter is better. `enhanced` should sit highest / flattest.

In [ ]:
from IPython.display import Image

from rl_nba.run import ExperimentOutcome, save_outputs

outcome = ExperimentOutcome(prepared=prepared, results=results, summary=summary)
metrics_path, plot_path = save_outputs(outcome, output_dir=PROJECT_ROOT / "examples" / "plots")
print("Saved:", metrics_path)
print("Saved:", plot_path)
Image(str(plot_path))

## That's the whole simulation

**What just happened:** the `enhanced` model — a contextual bandit that sees the full customer
view — learned, purely from trial-and-error feedback, to make offers worth far more than random
targeting, approaching the theoretical best.

**To change the experiment,** edit `config/rl_nba_config.yml` and re-run from the top:
- `reward.type` → `conversion`, `ape`, or `vnb`
- `products.catalog` → the product categories on offer
- `experiment.n_rounds` → longer runs = more learning
- `experiment.agents` → add or adjust the models being compared

*Remember: these customers are simulated. The results prove the machinery learns and that a
richer customer view pays off — not yet how real customers will behave.*